# 01 · Datos (Fase 1, Práctica 3)
Exploración y auditoría de KolektorSDD2, split de validación y pipeline de datos. **No se entrena nada.**

Los números que aparecen en las celdas son los reales de esta corrida: copiarlos de acá al avance de la Práctica 3, sin inventar ninguno.

Importante: las imágenes son **verticales** (~228 px de ancho × ~635 de alto). El tamaño de entrada es `(alto, ancho) = (608, 224)`.

In [ ]:
# --- Bootstrap: same cell at the top of every notebook ---
import os
import subprocess
import sys
from pathlib import Path

REPO = "matifleman/inspectia"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/InspectIA")
    REPO_DIR = Path("/content/inspectia")
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", f"https://github.com/{REPO}.git", str(REPO_DIR)], check=True)
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
else:
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DRIVE_ROOT = REPO_DIR
    if "WANDB_API_KEY" not in os.environ:
        os.environ["WANDB_MODE"] = "offline"  # local runs without a key log offline

os.chdir(REPO_DIR)
if IN_COLAB:  # locally, install once in your venv (see README)
    %pip install -q -r requirements.txt -e .
# Editable installs are only picked up after a kernel restart; this makes the
# package importable in the current session too.
sys.path.insert(0, str(REPO_DIR / "src"))

DATA_DIR = DRIVE_ROOT / "data"
CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints"
SPLITS_DIR = DRIVE_ROOT / "splits"

from inspectia.utils import get_device, set_seed

SEED = 42
set_seed(SEED)
device = get_device()
print(f"Repo: {REPO_DIR} | Data: {DATA_DIR} | Device: {device}")

In [ ]:
# Paths shared by every notebook after the bootstrap
DATASET_DIR = DATA_DIR / "KolektorSDD2"
SPLITS_DIR = REPO_DIR / "splits"            # versioned in git: the whole team uses the SAME splits
RESULTS_DIR = DRIVE_ROOT / "results"
FEATURES_DIR = DRIVE_ROOT / "features"
for d in (SPLITS_DIR, RESULTS_DIR, FEATURES_DIR, CHECKPOINT_DIR):
    d.mkdir(parents=True, exist_ok=True)

## 1. Dataset
Verifica los conteos; si ya está en `data/`, no descarga nada.

In [ ]:
!{sys.executable} scripts/download_data.py --dest "{DATA_DIR}"

In [ ]:
from inspectia.data import build_index, get_splits, get_transforms, make_loaders, IMG_SIZE
from inspectia import audit

df = build_index(DATASET_DIR)       # lee las 3.335 máscaras: ~1 min
audit.audit_counts(df)

## 2. Auditoría de calidad
Archivos corruptos, máscaras de otro tamaño que la imagen y duplicados exactos (sobre todo entre train y test: serían fuga).

In [ ]:
integrity = audit.check_integrity(df, DATASET_DIR)
duplicates = audit.find_duplicates(df, DATASET_DIR)
duplicates

## 3. Tamaño de los defectos
Este dato decide la resolución de entrada: si muchos defectos ocupan pocos píxeles, reducir la imagen los puede borrar.

In [ ]:
defects = audit.defect_summary(df)
audit.plot_defect_sizes(defects);

## 4. Iluminación
Si la brillantez media de algunas imágenes se aleja del resto, hay variación de iluminación.

In [ ]:
bright = audit.brightness_table(df, DATASET_DIR)
print(bright.groupby("label")["mean"].describe().round(1))
audit.plot_brightness(bright);

## 5. Ejemplos con máscara

In [ ]:
audit.plot_examples(df, DATASET_DIR, n=4);

## 6. Split de validación
Estratificado, semilla 42, guardado en `splits/*.csv` (versionado en git). Esperado: train 1864 (197 def.) · val 467 (49 def.) · test 1004 (110 def.).

In [ ]:
splits = get_splits(DATASET_DIR, SPLITS_DIR, val_frac=0.2, seed=SEED)
for name, d in splits.items():
    print(f"{name}: {len(d)} imágenes, {int(d['label'].sum())} defectuosas ({100 * d['label'].mean():.1f}%)")

## 7. Preprocesamiento y augmentation
Solo el train se aumenta. Figura: la misma defectuosa pasada 4 veces por el pipeline de train, para ver que el defecto no se sale del cuadro.

In [ ]:
import matplotlib.pyplot as plt, numpy as np
from PIL import Image
from inspectia.data import IMAGENET_MEAN, IMAGENET_STD

row = splits["train"].query("label == 1").iloc[0]
img = Image.open(DATASET_DIR / row["image"]).convert("RGB")
tf_train, tf_eval = get_transforms(True), get_transforms(False)
def show(t):
    x = t(img).numpy().transpose(1, 2, 0) * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    return np.clip(x, 0, 1)
fig, axes = plt.subplots(1, 5, figsize=(11, 6))
axes[0].imshow(show(tf_eval)); axes[0].set_title("val/test (sin aug.)")
for a in axes[1:]:
    a.imshow(show(tf_train)); a.set_title("train (aug.)")
for a in axes: a.axis("off")
print("shape de entrada:", tuple(tf_eval(img).shape), "= (canales, alto, ancho)")

## 8. DataLoaders
Un lote de cada split. `num_workers=0` acá para evitar problemas en el notebook de exploración.

In [ ]:
loaders = make_loaders(DATASET_DIR, splits, batch_size=16, num_workers=0)
for name, dl in loaders.items():
    x, y = next(iter(dl))
    print(f"{name}: lote {tuple(x.shape)} | defectuosas en el lote: {int(y.sum())} de {len(y)}")

## Avance de la Práctica 3 (redactar con los números de arriba)
- Fuente y licencia: KolektorSDD2 (ViCoS, Univ. de Ljubljana + Kolektor Group), CC BY-NC-SA 4.0.
- Conteos por split y clase, desbalance (≈ 11 % defectuosas), faltantes, duplicados y archivos corruptos (secciones 1-2).
- Tamaño de los defectos y su impacto en la resolución (sección 3); iluminación (sección 4).
- Riesgos de fuga y cómo se evitan: split oficial, validación estratificada del train, la máscara no es entrada, augmentation solo en train, sin duplicados entre splits.
- Plan de preparación: normalización ImageNet, entrada 608×224, augmentation seguro.